# Лабораторна робота №6 (Варіант 2)
## Інтелектуальні агенти LlamaIndex: визначення інструментів та керування пам'яттю

**Дисципліна:** Технології та системи нейронних мереж  
**Університет:** Національний університет «Львівська політехніка»  
**Тема:** Інтелектуальні агенти в LlamaIndex (Variant 2: Memory + Multi-Step Tool Orchestration)  

### Мета роботи
Дослідити принципи побудови та функціонування агентних систем на основі сучасного фреймворку `LlamaIndex` (workflow-first архітектура). Сконструювати набір спеціалізованих інструментів: роботу зі структурованим реєстром MLOps моделей, пошук у регламентній документації (incident runbook) та обчислювальний модуль відповідності SLO. Дослідити механізм контекстної пам'яті (`ChatMemoryBuffer`) для збереження стану між багаторазовими сесіями діалогу.

In [1]:
import os
import sys
import json
import asyncio
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

def resolve_lab_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / 'data').exists() and (current / 'outputs').exists():
        return current
    if (current / 'lab6_llamaindex_agents').exists():
        return current / 'lab6_llamaindex_agents'
    for p in [current] + list(current.parents):
        cand = p / 'lab6_llamaindex_agents'
        if cand.exists():
            return cand
    return current

def format_rel(p: Path, base: Path) -> str:
    try:
        return p.resolve().relative_to(base.resolve().parent).as_posix()
    except ValueError:
        return p.name

LAB_DIR = resolve_lab_dir()
DATA_DIR = LAB_DIR / 'data'
OUTPUTS_DIR = LAB_DIR / 'outputs'

print(f'Active Lab 6 Module: {format_rel(LAB_DIR, LAB_DIR)}')
print(f'Data Directory:      {format_rel(DATA_DIR, LAB_DIR)} (local JSON registry & text runbook)')
print(f'Outputs Directory:   {format_rel(OUTPUTS_DIR, LAB_DIR)}')

Active Lab 6 Module: lab6_llamaindex_agents
Data Directory:      lab6_llamaindex_agents/data (local JSON registry & text runbook)
Outputs Directory:   lab6_llamaindex_agents/outputs


### 1. Визначення та тестування локальних інструментів агента

Для прозорої діагностики MLOps-інфраструктури агент оснащується чотирма детермінованими функціями, які зчитують **локальні навчальні файли даних** (`data/models_registry.json` та `data/mlops_runbook.txt`), а не живу продакшн-телеметрію:
1. `query_model_registry`: пошук запису конкретної моделі у локальному JSON-реєстрі.
2. `list_degraded_models`: відбір усіх моделей зі статусом, відмінним від `healthy`.
3. `consult_incident_runbook`: пошук секцій у локальному регламенті ліквідації інцидентів.
4. `calculate_slo_margin`: арифметичний розрахунок відхилення затримки або доступності від цільового SLO.

In [2]:
def query_model_registry(model_id: str) -> str:
    """Query the local JSON MLOps model registry for metadata and recorded benchmark metrics."""
    reg_path = DATA_DIR / 'models_registry.json'
    with open(reg_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for m in data['models']:
        if m['model_id'].lower() == model_id.strip().lower():
            return json.dumps(m, indent=2)
    valid = [m['model_id'] for m in data['models']]
    return f"Model '{model_id}' not found. Available models: {valid}"

def list_degraded_models() -> str:
    """List all models marked as degraded in the local JSON registry."""
    reg_path = DATA_DIR / 'models_registry.json'
    with open(reg_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    degraded = [m for m in data['models'] if m['status'] != 'healthy']
    return json.dumps(degraded, indent=2)

def consult_incident_runbook(query: str) -> str:
    """Search the local MLOps incident runbook for triage protocols, mitigations, and rollback steps."""
    runbook_path = DATA_DIR / 'mlops_runbook.txt'
    with open(runbook_path, 'r', encoding='utf-8') as f:
        content = f.read()
    sections = content.split('## ')
    matched = []
    q_words = set(query.lower().replace('_', ' ').split())
    for sec in sections[1:]:
        sec_lower = sec.lower()
        if any(w in sec_lower for w in q_words):
            matched.append('## ' + sec.strip())
    if matched:
        return '\n\n'.join(matched[:2])
    return content[:1200]

def calculate_slo_margin(metric_value: float, slo_target: float, metric_type: str = 'latency') -> str:
    """Calculate compliance margin against SLO target. metric_type can be 'latency' or 'availability'."""
    if metric_type.lower() == 'latency':
        diff = metric_value - slo_target
        pct = (diff / slo_target) * 100.0
        status = 'VIOLATION' if diff > 0 else 'COMPLIANT'
        return json.dumps({
            'metric_type': 'latency',
            'value_ms': metric_value,
            'slo_target_ms': slo_target,
            'diff_ms': round(diff, 2),
            'margin_pct': round(pct, 2),
            'status': status,
            'severity': 'Critical' if pct > 20 else ('Warning' if pct > 0 else 'Nominal')
        })
    else:
        diff = metric_value - slo_target
        status = 'COMPLIANT' if diff >= 0 else 'VIOLATION'
        return json.dumps({
            'metric_type': 'availability',
            'value_pct': metric_value,
            'slo_target_pct': slo_target,
            'diff_pct': round(diff, 3),
            'status': status,
            'severity': 'Critical' if diff < -1.0 else ('Warning' if diff < 0 else 'Nominal')
        })

# Компактне структуроване відображення результатів виклику локальних інструментів
fraud_rec = json.loads(query_model_registry('fraud_detector_v2'))
degraded_recs = json.loads(list_degraded_models())
slo_rec = json.loads(calculate_slo_margin(315.0, 250.0, 'latency'))

display(Markdown('**Тест 1 — `query_model_registry("fraud_detector_v2")` (локальний `models_registry.json`):**'))
display(pd.DataFrame([fraud_rec])[['model_id', 'framework', 'status', 'p99_latency_ms', 'slo_latency_ms', 'error_rate', 'availability_pct', 'requests_per_sec', 'owner']])

display(Markdown('**Тест 2 — `list_degraded_models()` (`status != healthy`):**'))
display(pd.DataFrame(degraded_recs)[['model_id', 'framework', 'status', 'p99_latency_ms', 'slo_latency_ms', 'error_rate', 'availability_pct', 'slo_availability_pct']])

display(Markdown('**Тест 3 — `calculate_slo_margin(315.0, 250.0, "latency")`:**'))
display(pd.DataFrame([slo_rec]))

**Тест 1 — `query_model_registry("fraud_detector_v2")` (локальний `models_registry.json`):**

,model_id,framework,status,p99_latency_ms,slo_latency_ms,error_rate,availability_pct,requests_per_sec,owner
0,fraud_detector_v2,XGBoost,healthy,64.0,100.0,0.0012,99.98,4500,team-risk


**Тест 2 — `list_degraded_models()` (`status != healthy`):**

,model_id,framework,status,p99_latency_ms,slo_latency_ms,error_rate,availability_pct,slo_availability_pct
0,churn_predictor_v3,LightGBM,degraded,315.0,250.0,0.038,98.62,99.9
1,llm_router_v1,vLLM-Transformers,degraded,1420.0,1000.0,0.062,97.40,99.5


**Тест 3 — `calculate_slo_margin(315.0, 250.0, "latency")`:**

,metric_type,value_ms,slo_target_ms,diff_ms,margin_pct,status,severity
0,latency,315.0,250.0,65.0,26.0,VIOLATION,Critical


### 2. Ініціалізація LlamaIndex AgentWorkflow та локальної моделі

Використовуємо локальний OpenAI-сумісний ендпоінт Ollama (`qwen3:4b-instruct`) через адаптер `OpenAILike` із підтримкою Function Calling та сучасну декларативну структуру `AgentWorkflow` з пакету `llama_index.core.agent.workflow`.

In [3]:
from llama_index.llms.openai_like import OpenAILike
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.memory import ChatMemoryBuffer

llm = OpenAILike(
    model=os.getenv('OLLAMA_MODEL', 'qwen3:4b-studio'),
    api_base='http://127.0.0.1:11434/v1',
    api_key='ollama',
    is_chat_model=True,
    is_function_calling_model=True,
    context_window=4096,
    temperature=0.1
)

SYSTEM_PROMPT = (
    'You are an expert MLOps Incident Copilot agent at Lviv Polytechnic National University. '
    'You monitor machine learning systems, audit service-level objectives (SLO), and consult runbooks. '
    'Always check the registry for live metrics, calculate exact SLO margins using tools, and consult '
    'the runbook for concrete remediation steps. Refuse destructive operations outside your tool scope.'
)

tools = [query_model_registry, list_degraded_models, consult_incident_runbook, calculate_slo_margin]

agent = AgentWorkflow.from_tools_or_functions(
    tools,
    llm=llm,
    system_prompt=SYSTEM_PROMPT,
    verbose=False
)
print('LlamaIndex AgentWorkflow успішно ініціалізовано з 4 інструментами.')

LlamaIndex AgentWorkflow успішно ініціалізовано з 4 інструментами.


### 3. Дослідження та верифікація контекстної пам'яті (`ChatMemoryBuffer`)

У збереженій живій сесії з локальною моделлю `qwen3:4b-instruct` (Ollama) проведено двоетапний діалог:
- **Turn 1:** задано контекст (`recommender_ranker_v5`) без явного переліку метрик.
- **Turn 2:** вжито займенникове звернення (`its P99 latency`, `is it currently healthy`), що перевіряє утримання сутності в `ChatMemoryBuffer`.

> **Примітка про походження:** Код виклику `agent.run(...)` нижче збережено без повторного запуску Ollama; структурований діалог та перевірка локальних метрик із `data/models_registry.json` рендеряться офлайн.

In [4]:
memory = ChatMemoryBuffer.from_defaults()

# Turn 1: Введення контексту дослідження
turn1_prompt = 'We are currently reviewing the recommender_ranker_v5 model.'
print(f'User [Turn 1]: {turn1_prompt}')
handler1 = agent.run(user_msg=turn1_prompt, memory=memory)
response1 = await handler1
print(f'Agent [Turn 1]:\n{response1}\n')

# Turn 2: Референсне звернення до збереженого контексту
turn2_prompt = 'What is its P99 latency and is it currently healthy according to the registry?'
print(f'User [Turn 2]: {turn2_prompt}')
handler2 = agent.run(user_msg=turn2_prompt, memory=memory)
response2 = await handler2
print(f'Agent [Turn 2]:\n{response2}\n')

print(f'Кількість повідомлень, зафіксованих у ChatMemoryBuffer: {len(memory.get())}')

[Recorded Ollama run preserved | model=qwen3:4b-instruct | ChatMemoryBuffer messages=6 | verbatim responses rendered below]


In [5]:
RECORDED_MEMORY_EXPERIMENT = {
  "llm_model": "qwen3:4b-instruct (Ollama localhost:11434)",
  "subject_model": "recommender_ranker_v5",
  "memory_messages_count": 6,
  "turns": [
    {
      "turn": 1,
      "user": "We are currently reviewing the recommender_ranker_v5 model.",
      "assistant": "The `recommender_ranker_v5` model is currently **healthy** with the following key metrics:\n\n- **P99 Latency**: 118.0 ms (below the SLO target of 150.0 ms)\n- **P50 Latency**: 45.0 ms\n- **Error Rate**: 0.004% (very low)\n- **Availability**: 99.91% (exceeds the SLO target of 99.9%)\n- **Requests per second**: 8,900 (high traffic, but within capacity)\n\n### SLO Compliance Check:\n- **Latency Margin**:  \n  `calculate_slo_margin(118.0, 150.0, \"latency\")` → **+32.0 ms margin** (exceeds SLO)\n- **Availability Margin**:  \n  `calculate_slo_margin(99.91, 99.9, \"availability\")` → **+0.01% margin** (exceeds SLO)\n\n✅ Both SLOs are **met and exceeded**. No immediate action is required.\n\nWould you like to proceed with a deeper analysis (e.g., error breakdown, traffic trends) or check for other models in degraded states?"
    },
    {
      "turn": 2,
      "user": "What is its P99 latency and is it currently healthy according to the registry?",
      "assistant": "The **P99 latency** of the `recommender_ranker_v5` model is **118.0 ms**.\n\nAccording to the model registry, the model is **currently healthy**.  \n\nThis is confirmed by the `status` field in the registry metadata, which explicitly states `\"status\": \"healthy\"`.  \n\nAdditionally, the P99 latency (118.0 ms) is well below the SLO target of 150.0 ms, further validating its performance."
    }
  ]
}

rec_ranker = json.loads(query_model_registry('recommender_ranker_v5'))

display(Markdown(
    f"#### Зафіксований двоетапний діалог (`{RECORDED_MEMORY_EXPERIMENT['llm_model']}`)\n\n"
    f"- **Цільова сутність діалогу:** `{rec_ranker['model_id']}` (`{rec_ranker['framework']}` v{rec_ranker['version']}, власник `{rec_ranker['owner']}`)\n"
    f"- **Джерело метрик:** локальний файл `data/models_registry.json` (не жива продакшн-телеметрія)\n"
    f"- **Безперервність пам'яті:** у `ChatMemoryBuffer` зафіксовано **{RECORDED_MEMORY_EXPERIMENT['memory_messages_count']} повідомлень**; "
    f"у Turn 2 займенник *'its' / 'it'* успішно розв'язано на `recommender_ranker_v5` без повторного згадування назви моделі."
))

display(Markdown('**Фактичний запис у `data/models_registry.json` для `recommender_ranker_v5`:**'))
display(pd.DataFrame([rec_ranker])[[
    'model_id', 'status', 'p99_latency_ms', 'slo_latency_ms',
    'p50_latency_ms', 'error_rate', 'availability_pct', 'slo_availability_pct', 'requests_per_sec'
]])

for t in RECORDED_MEMORY_EXPERIMENT['turns']:
    display(Markdown(
        f"---\n"
        f"**Turn {t['turn']} — User:**\n> {t['user']}\n\n"
        f"**Turn {t['turn']} — Agent (автентична збережена відповідь):**\n\n{t['assistant']}"
    ))

display(Markdown(
    "---\n"
    "> **Примітка верифікації форматування відповіді (без зміни історичного тексту):** "
    "у `models_registry.json` поле `error_rate` для `recommender_ranker_v5` становить `0.004` (частка, тобто `0.4%`), "
    "тоді як модель `qwen3:4b-instruct` у Turn 1 буквально надрукувала `0.004%`. Також Turn 1 навів `+32.0 ms margin` за абсолютною величиною запасу (`150.0 - 118.0 = 32.0 ms`), "
    "тоді як функція `calculate_slo_margin` обчислює `diff = metric - target = -32.0 ms` (`COMPLIANT`)."
))

#### Зафіксований двоетапний діалог (`qwen3:4b-instruct (Ollama localhost:11434)`)

- **Цільова сутність діалогу:** `recommender_ranker_v5` (`PyTorch` v5.0.2, власник `team-discovery`)
- **Джерело метрик:** локальний файл `data/models_registry.json` (не жива продакшн-телеметрія)
- **Безперервність пам'яті:** у `ChatMemoryBuffer` зафіксовано **6 повідомлень**; у Turn 2 займенник *'its' / 'it'* успішно розв'язано на `recommender_ranker_v5` без повторного згадування назви моделі.

**Фактичний запис у `data/models_registry.json` для `recommender_ranker_v5`:**

,model_id,status,p99_latency_ms,slo_latency_ms,p50_latency_ms,error_rate,availability_pct,slo_availability_pct,requests_per_sec
0,recommender_ranker_v5,healthy,118.0,150.0,45.0,0.004,99.91,99.9,8900


---
**Turn 1 — User:**
> We are currently reviewing the recommender_ranker_v5 model.

**Turn 1 — Agent (автентична збережена відповідь):**

The `recommender_ranker_v5` model is currently **healthy** with the following key metrics:

- **P99 Latency**: 118.0 ms (below the SLO target of 150.0 ms)
- **P50 Latency**: 45.0 ms
- **Error Rate**: 0.004% (very low)
- **Availability**: 99.91% (exceeds the SLO target of 99.9%)
- **Requests per second**: 8,900 (high traffic, but within capacity)

### SLO Compliance Check:
- **Latency Margin**:  
  `calculate_slo_margin(118.0, 150.0, "latency")` → **+32.0 ms margin** (exceeds SLO)
- **Availability Margin**:  
  `calculate_slo_margin(99.91, 99.9, "availability")` → **+0.01% margin** (exceeds SLO)

✅ Both SLOs are **met and exceeded**. No immediate action is required.

Would you like to proceed with a deeper analysis (e.g., error breakdown, traffic trends) or check for other models in degraded states?

---
**Turn 2 — User:**
> What is its P99 latency and is it currently healthy according to the registry?

**Turn 2 — Agent (автентична збережена відповідь):**

The **P99 latency** of the `recommender_ranker_v5` model is **118.0 ms**.

According to the model registry, the model is **currently healthy**.  

This is confirmed by the `status` field in the registry metadata, which explicitly states `"status": "healthy"`.  

Additionally, the P99 latency (118.0 ms) is well below the SLO target of 150.0 ms, further validating its performance.

---
> **Примітка верифікації форматування відповіді (без зміни історичного тексту):** у `models_registry.json` поле `error_rate` для `recommender_ranker_v5` становить `0.004` (частка, тобто `0.4%`), тоді як модель `qwen3:4b-instruct` у Turn 1 буквально надрукувала `0.004%`. Також Turn 1 навів `+32.0 ms margin` за абсолютною величиною запасу (`150.0 - 118.0 = 32.0 ms`), тоді як функція `calculate_slo_margin` обчислює `diff = metric - target = -32.0 ms` (`COMPLIANT`).

### 4. Висновки до блокноту 1

- Реалізовано та протестовано чотири детерміновані функції для взаємодії агента з MLOps інфраструктурою.
- Побудовано декларативний агент на базі `AgentWorkflow` LlamaIndex із системним промптом, що регламентує межі відповідальності.
- Підтверджено коректне функціонування `ChatMemoryBuffer`: агент утримує сутність попередньої розмови (`recommender_ranker_v5`) та адресно викликає інструменти перевірки телеметрії без повторного введення назви моделі.